In [ ]:
# import json
# import re

# lst=[]
# with open ('../../code/3/jawiki-country.json','r',encoding='utf-8_sig') as f:
#     for line in f.readlines():
#         dic=json.loads(line)
#         lst.append(dic)

# texts = []
# for content in lst:
#     if content['title'] == '日本':
#         texts.append(content["text"])

# output_texts = []
# for t in texts:
#     output_texts.append(re.sub(r'\'*|\[*|\]*|{*|}*|=*|\||<!--(.*?)-->',"",t))

# with open("../../code/4/Japan.txt",'a') as f_:
#     for text in output_texts:
#         print(text,file=f_)

In [6]:
import MeCab
import re

# ファイル読み込み
with open("../../code/4/kokoro.txt", "r") as f:
    content = f.read()

# MeCabの解析器
tagger = MeCab.Tagger()

# （単語, 品詞）ペアを格納
word_pos_list = []

# 形態素解析
content_ = re.split(r'\n|。',content)
output = []
for sentence in content_:
    sentence = re.sub(' ','',sentence)
    tmp = ""
    node = tagger.parseToNode(sentence)
    while node:
        surface = node.surface
        features = node.feature.split(',')
        pos = features[0]  # 品詞

        if surface:
            tmp += surface + ' '

        node = node.next

    output.append(tmp)

with open("../../code/4/tokenized_kokoro.txt",'a') as f_:
    for text in output:
        print(text,file=f_)

In [19]:
import MeCab
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import numpy as np

# MeCab の用意
wakati_tagger = MeCab.Tagger("-Owakati")
node_tagger   = MeCab.Tagger()

def tokenize(text):
    """全文を MeCab で分かち書きしてトークンリストを返す"""
    return wakati_tagger.parse(text).strip().split()

def is_noun(word):
    """表層形が名詞かどうかを MeCab で判定する"""
    node = node_tagger.parseToNode(word)
    while node:
        if node.surface == word:
            return node.feature.split(",")[0] == "名詞"
        node = node.next
    return False

# コーパス読み込み
with open("tokenized_kokoro.txt", "r", encoding="utf-8") as f:
    corpus = [line.strip() for line in f]

# TF-IDF 計算（全文トークンを使う）
vectorizer = TfidfVectorizer(tokenizer=tokenize, token_pattern=None)
X       = vectorizer.fit_transform(corpus).toarray()  # shape=(n_docs, n_terms)
terms   = vectorizer.get_feature_names_out()          # 単語リスト
idf     = vectorizer.idf_                            # 各単語の IDF 値

# TF を再計算 (TF-IDF ÷ IDF)
tf      = X / idf

# 各単語の「平均TF」「IDF」「平均TF-IDF」をまとめる
mean_tf     = tf.mean(axis=0)
mean_tfidf  = X.mean(axis=0)

df = pd.DataFrame({
    "term":    terms,
    "TF":      mean_tf,
    "IDF":     idf,
    "TF-IDF":  mean_tfidf
})

# 名詞だけ抽出 → TF-IDF 上位20件を表示
df_nouns = df[df["term"].apply(is_noun)]
top20    = df_nouns.sort_values("TF-IDF", ascending=False).head(20)

print(top20.reset_index(drop=True))


   term        TF       IDF    TF-IDF
0    先生  0.004872  3.404040  0.016586
1     事  0.004130  3.400478  0.014044
2     奥  0.003103  3.760332  0.011668
3     ｋ  0.003096  3.755250  0.011627
4    もの  0.002649  3.799298  0.010064
5     父  0.002202  4.061265  0.008942
6    あり  0.001697  4.314549  0.007323
7    自分  0.001708  4.245999  0.007253
8    うち  0.001587  4.279687  0.006791
9     一  0.001447  4.457309  0.006449
10    母  0.001399  4.548280  0.006363
11    人  0.001415  4.437055  0.006277
12    方  0.001358  4.457309  0.006053
13    気  0.001299  4.542694  0.005902
14    嬢  0.001197  4.617874  0.005529
15    顔  0.001043  4.846286  0.005057
16    出  0.001057  4.759589  0.005031
17    今  0.001044  4.692675  0.004900
18    前  0.001058  4.611886  0.004879
19   答え  0.000935  5.216530  0.004875


In [20]:
import MeCab
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import numpy as np

# MeCab の用意
wakati_tagger = MeCab.Tagger("-Owakati")
node_tagger   = MeCab.Tagger()

def tokenize(text):
    """全文を MeCab で分かち書きしてトークンリストを返す"""
    return wakati_tagger.parse(text).strip().split()

def is_noun(word):
    """表層形が名詞かどうかを MeCab で判定する"""
    node = node_tagger.parseToNode(word)
    while node:
        if node.surface == word:
            return node.feature.split(",")[0] == "名詞"
        node = node.next
    return False

# コーパス読み込み
with open("tokenized_kokoro.txt", "r", encoding="utf-8") as f:
    corpus = [line.strip() for line in f]

# TF-IDF 計算（全文トークンを使う）
vectorizer = TfidfVectorizer(tokenizer=tokenize, token_pattern=None)
X       = vectorizer.fit_transform(corpus).toarray()  # shape=(n_docs, n_terms)
terms   = vectorizer.get_feature_names_out()          # 単語リスト
idf     = vectorizer.idf_                            # 各単語の IDF 値

# TF を再計算 (TF-IDF ÷ IDF)
tf      = X / idf

# 各単語の「平均TF」「IDF」「平均TF-IDF」をまとめる
mean_tf     = tf.mean(axis=0)
mean_tfidf  = X.mean(axis=0)

df = pd.DataFrame({
    "term":    terms,
    "TF":      mean_tf,
    "IDF":     idf,
    "TF-IDF":  mean_tfidf
})

# 名詞だけ抽出 → TF-IDF 上位20件を表示
top20 = df.sort_values("TF-IDF", ascending=False).head(20)

print(top20.reset_index(drop=True))


   term        TF       IDF    TF-IDF
0     の  0.039440  1.644154  0.064845
1     た  0.040655  1.448743  0.058898
2     に  0.030545  1.723007  0.052629
3     《  0.028018  1.806808  0.050623
4     》  0.028018  1.806808  0.050623
5     は  0.032203  1.558025  0.050174
6     て  0.025523  1.851498  0.047255
7     を  0.021665  1.933731  0.041894
8     、  0.021287  1.957803  0.041676
9     私  0.020542  1.934141  0.039732
10    と  0.019258  2.043727  0.039358
11    が  0.015426  2.228202  0.034373
12    も  0.013462  2.398866  0.032293
13    で  0.012737  2.385891  0.030389
14    し  0.011937  2.447705  0.029217
15    い  0.008936  2.740313  0.024488
16   です  0.008567  2.753258  0.023586
17   まし  0.008205  2.810718  0.023062
18    「  0.006833  3.182503  0.021747
19   ない  0.007451  2.898222  0.021594
